# 5.13 資料或模型變大，哪個值得花預算？


模型變大、資料也從16篇換64篇，成績提升後很難知道哪個改動有用。先做兩寬度×兩資料量的四格設計，讓一條比較只改模型，另一條只改資料。

計畫用寬度8或16、不同記錄16或64，每步四筆、每筆八個有效答案、100步。每格都是3200個答案曝光；16篇會重複更多次，64篇則有更多不同內容。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
from tiny_perceptron.model import TinyLM, ModelConfig

for width in [8, 16]:
    parameters = TinyLM(ModelConfig(width=width)).description()["parameters"]
    for records in [16, 64]:
        batch_size, answers_per_record, steps = 4, 8, 100
        print(
            {
                "width": width,
                "不同記錄": records,
                "參數格數": parameters,
                "更新步數": steps,
                "有效答案預算": batch_size * answers_per_record * steps,
                "品質": "尚待訓練與量測",
            }
        )

輸入是實驗計劃，不是現成的訓練資料。`records` 指不同例子數，batch_size是每步四筆，假定每筆八個有效答案，所以每格共3200答案；16筆資料會被重複更多次，64筆則包含更多不同題。程序只建立模型數參數並印四份計畫，品質欄保留未測狀態，不能從參數數字推定分數。

「公平預算」有多種定義。固定步數方便看每次更新效果；固定有效token數讓見到的監督總量相近；固定時間或FLOPs則更直接限制計算成本。FLOPs是浮點運算次數，例如一次乘法與一次加法各算一個。D是輸入特徵數，H是輸出特徵數；密集Linear用一張H橫排、D欄的權重表，把D個輸入數加權混合成H個輸出數，每個輸出都使用全部輸入。處理一個位置約需2×D×H次運算，這裡忽略每個輸出混合完再加上一個可調數字的成本（這個數字叫偏置），以及其他較小成本：8進8出約128次，16進16出約512次。模型變寬會同時放大兩邊，所以相同token預算也不等於相同計算預算。耗時測量可回看 [5.9](https://birdhackor.github.io/tiny-perceptron-vlm/5.9.html)。

看四格實驗的品質時，訓練代價與驗證代價指的是「平均下一文字單位猜錯代價」，也就是 loss：用相應資料的有效答案重算、取平均，計算規則回看[5.2](https://birdhackor.github.io/tiny-perceptron-vlm/5.2.html)。已有實驗中，更多材料的驗證代價較低、訓練代價較高；訓練平均覆蓋不同材料，不能當同一批新題成績。

固定步數、有效token、秒數或運算量各回答不同預算問題。比較前選主要限制，再用共同新題和實際成本判讀；四個小設定不足以外推大型模型規模規律。

練習保持每步四筆，假設某格每筆有效答案只有四個，想維持3200token預算應跑幾步？先算 `3200/(4×4)=200`，再改該格記錄核對。若仍寫100步，那格只有1600答案，已經不是同樣監督預算。將計劃中的假設換成實測長度統計，纔會成為真實資源記錄。

<details>
<summary>補充：實作約定與原始紀錄</summary>

把計畫落成實驗，結果不必沿用上面的示意數字。我們已另外產生80篇`object=67;color=red;shape=circle.`這類短文，先按物體編號留出8篇驗證與8篇test，再從其餘64篇中取16篇或全部64篇練習。模型寬度改用16或32，各一層，種子42；四組都更新150次、每次4篇。寬度16的模型有13,744個參數，寬度32則有33,632個。下面是實際測得的四格，代價都是更新完成後重算的平均下一文字單位代價：

| 模型寬度 | 不同練習短文 | 累計有效目標 | 訓練代價 | 驗證代價 | test代價 |
| --- | ---: | ---: | ---: | ---: | ---: |
| 16 | 16篇 | 20,765 | 0.27166 | 0.49288 | 0.39779 |
| 16 | 64篇 | 20,623 | 0.33032 | 0.31105 | 0.31061 |
| 32 | 16篇 | 20,765 | 0.14502 | 0.40927 | 0.27961 |
| 32 | 64篇 | 20,623 | 0.20107 | 0.21914 | 0.22258 |

這一次，兩種寬度下，64篇的驗證代價都比16篇低，訓練代價卻較高。要記得，兩組訓練平均數涵蓋的是不同的材料，不能把它們當成在同一批新題的成績。固定150步也不等於所有材料都練同樣多次，16篇會被反覆抽到更多。有效目標20,765與20,623並不相同，因為實際抽到的短文長度不同；我們匹配的是更新次數與每批筆數，沒有把監督總量或計算成本控制成完全相等。

較大模型與更多資料如何共同影響品質，常被研究成scaling law（規模與表現的經驗規律）。眼前只有一個種子、有限的規則短文、四個小設定，能回答的是這批資料在這個更新預算下如何變化，不能由此說64篇必然較好，更不能外推大型模型的規律。[實測原始報告](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/text_foundation.json)保留四組資料與成本；重做入口在[T.4](https://birdhackor.github.io/tiny-perceptron-vlm/training.html#T.4)。

短程式沿用本課工具與原計算語義。安裝、長訓練與重做操作見[訓練配方](https://birdhackor.github.io/tiny-perceptron-vlm/training.html)，不需要先完成長配方才能閱讀這個例子。

</details>
